# Análise de colisões canônicas

Estudo exploratório, somente leitura, sobre `desafio1_bracis.db`. O parser
estrutural do Notebook 04 é reutilizado sem alteração. Esta etapa testa a
seletividade de `(tribunal, numero_normalizado)` e reúne evidências para
auditoria posterior; não decide se dois registros são o mesmo feito e não
implementa `CanonicalIndex`.

# 1. Setup e parser congelado

O Notebook 04 é a fonte congelada da extração estrutural validada. Esta
análise apenas carrega seu resultado, sem alterar ou reabrir os parsers.
O banco é aberto em modo read-only pela função do projeto.

In [ ]:
import hashlib
import re

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from bracis_jusbrasil.database import connect_database, get_database_path

connection = connect_database(get_database_path())
TRIBUNAIS = ('STF', 'STJ', 'TSE', 'TST', 'STM')
LIMITES = (250, 500, 1000, 2500, 5000, 10000)

def frame(sql, params=()):
    return pd.read_sql_query(sql, connection, params=params)

def compactar(texto):
    return ' '.join((texto or '').split())

def preview(texto, limit=420):
    texto = compactar(texto)
    return texto[:limit] + ('…' if len(texto) > limit else '')

def normalizar_numero(value):
    return re.sub(r'[^0-9]', '', value) if value else None

acordaos = frame('''SELECT documento_id, id, tribunal, ano, relator, texto
                   FROM documentos
                   WHERE natureza = 'acordao'
                   ORDER BY documento_id''')
assert len(acordaos) == 1000
assert acordaos.groupby('tribunal').size().eq(200).all()
acordaos['texto_compacto'] = acordaos.texto.map(compactar)
print(acordaos.groupby('tribunal').size().to_string())

In [ ]:
CNJ = re.compile(
    r'(?<!\d)\d{3,7}\s*-\s*\d{2}\s*[. ]\s*\d{4}\s*[. ]\s*\d\s*[. ]\s*\d{2}\s*[. ]\s*\d{4}(?:\s*/\s*[A-Z]{2})?(?!\d)', re.I)
STF_NUMBER = re.compile(r'(?<!\d)\d{1,3}(?:\.\d{3})+(?!\d)')
STJ_TITLE = re.compile(
    r'\b(?P<class>(?:RECURSO|AGRAVO|AGINT|HABEAS|EMBARGOS|MANDADO|AÇÃO|CONFLITO|CAUTELAR|QO|RECLAMAÇÃO|SUSPENSÃO|PETIÇÃO|EDCL)[^.;]{0,140}?)'
    r'\s+N[ºO.]\s*(?P<num>\d{1,3}(?:\.\d{3})*|\d+)\s*-\s*(?P<uf>[A-Z]{2})\b', re.I)
TST_NUMBER = re.compile(
    r'(?P<raw>(?:TST\s*-\s*)?[A-Z][A-Z0-9]*(?:\s*-\s*[A-Z][A-Z0-9]*)*\s*-\s*'
    r'\d{1,7}\s*-\s*\d{2}\s*[. ]\s*\d{4}\s*[. ]\s*\d\s*[. ]\s*\d{2}\s*[. ]\s*\d{4})', re.I)
STM_TITLE = re.compile(
    r'\b(?P<class>(?:APELA[ÇC][AÃ]O|AGRAVO|EMBARGOS|HABEAS CORPUS|CONFLITO|RECURSO|'
    r'A[ÇC][AÃ]O|CORREI[ÇC][AÃ]O|REPRESENTA[ÇC][AÃ]O|RECLAMA[ÇC][AÃ]O)[^,]{0,140}?)'
    r'\s+N[ºO.]\s*(?P<num>\d{3,7}\s*-\s*\d{2}\s*[. ]\s*\d{4}\s*[. ]\s*'
    r'\d\s*[. ]\s*\d{2}\s*[. ]\s*\d{4})(?:\s*/\s*[A-Z]{2})?', re.I)

def make_record(pattern, start=None, block=None, raw=None, number_pos=None, status='unparsed'):
    return {'identity_pattern': pattern, 'identity_block': block,
            'identity_block_start': start, 'numero_raw': raw,
            'numero_normalizado': normalizar_numero(raw),
            'numero_posicao': number_pos, 'parse_status': status}

def finish_record(pattern, text, start, end, matches, number_match, raw=None):
    block = text[start:end]
    values = {normalizar_numero(m.group(0)) for m in matches}
    if not matches:
        return make_record(pattern, start, block, status='suspicious')
    if len(values) > 1:
        return make_record(pattern, start, block, status='ambiguous')
    raw = raw or number_match.group(0)
    return make_record(pattern, start, block, raw, number_match.start(), 'extracted')

def parse_stf_primary_number(text):
    s = compactar(text)
    relator = re.search(r'\bRELATORA?\b', s[:5000], re.I)
    if not relator:
        return make_record('STF: órgão + classe + número + relator')
    anchors = list(re.finditer(r'\b(?:PRIMEIRA|SEGUNDA|TERCEIRA|QUARTA|QUINTA|SEXTA) TURMA\b|\bPLEN[ÁA]RIO\b', s[:relator.start()], re.I))
    start = anchors[-1].start() if anchors else 0
    matches = list(STF_NUMBER.finditer(s, start, relator.start()))
    return finish_record('STF: órgão + classe + número + relator', s, start,
                        min(len(s), relator.end() + 120), matches,
                        matches[-1] if matches else None)

def parse_stj_primary_number(text):
    s = compactar(text)
    matches = [m for m in STJ_TITLE.finditer(s[:2200])
               if re.search(r'\bRELATORA?\b', s[m.end():m.end() + 260], re.I)]
    if not matches:
        return make_record('STJ: classe + Nº + número + UF + relator')
    m = matches[0]
    relator = re.search(r'\bRELATORA?\b', s[m.end():m.end() + 260], re.I)
    return finish_record('STJ: classe + Nº + número + UF + relator', s, m.start(),
                        min(len(s), m.end() + relator.end() + 120), matches, m,
                        m.group('num'))

def parse_tse_primary_number(text):
    s = compactar(text)
    institution = re.search(r'TR(?:IB|IE)UNAL SUPERIOR ELEITORAL', s[:3000], re.I)
    if not institution:
        return make_record('TSE: instituição + acórdão + classe + CNJ')
    acordao = re.search(r'AC[ÓO€]RD[ÃA]O', s[institution.end():institution.end() + 1200], re.I)
    if not acordao:
        return make_record('TSE: instituição + acórdão + classe + CNJ')
    start = institution.start()
    number_start = institution.end() + acordao.end()
    relator = re.search(r'\b(?:RELATOR|REIATOR)\b', s[number_start:number_start + 1400], re.I)
    end = number_start + relator.start() if relator else min(len(s), number_start + 1200)
    matches = list(CNJ.finditer(s, number_start, end))
    return finish_record('TSE: instituição + acórdão + classe + CNJ', s, start,
                        min(len(s), end + 100), matches,
                        matches[0] if matches else None)

def parse_tst_primary_number(text):
    s = compactar(text)
    anchor = re.search(r'VISTOS,? RELATADOS E DISCUTIDOS ESTES AUTOS', s, re.I)
    if not anchor:
        return make_record('TST: Vistos/autos + identificador TST/CNJ')
    tail = s[anchor.end():anchor.end() + 1300]
    end_marker = re.search(r',\s*EM QUE\b', tail, re.I)
    end = anchor.end() + end_marker.start() if end_marker else min(len(s), anchor.end() + 1000)
    matches = list(TST_NUMBER.finditer(s, anchor.end(), end))
    return finish_record('TST: Vistos/autos + identificador TST/CNJ', s, anchor.start(),
                        min(len(s), end + 120), matches,
                        matches[0] if matches else None)

def parse_stm_primary_number(text):
    s = compactar(text)
    relator = re.search(r'\bRELATORA?\b', s[:5500], re.I)
    limite = relator.start() if relator else 5500
    matches = list(STM_TITLE.finditer(s[:limite]))
    if not matches:
        return make_record('STM: classe + Nº + CNJ + UF')
    m = matches[0]
    matches = [m]
    return finish_record('STM: classe + Nº + CNJ + UF', s, m.start(),
                        min(len(s), m.end() + 180), matches, m, m.group('num'))

PARSERS = {'STF': parse_stf_primary_number, 'STJ': parse_stj_primary_number,
           'TSE': parse_tse_primary_number, 'TST': parse_tst_primary_number,
           'STM': parse_stm_primary_number}
parsed = pd.DataFrame([PARSERS[r.tribunal](r.texto) for r in acordaos.itertuples()])
resultado = pd.concat([acordaos.reset_index(drop=True), parsed], axis=1)
resultado['identity_block_end'] = resultado.apply(
    lambda r: r.identity_block_start + len(r.identity_block)
    if isinstance(r.identity_block, str) and r.identity_block_start is not None else None, axis=1)
display(resultado[['tribunal', 'documento_id', 'identity_pattern',
                   'identity_block_start', 'numero_raw', 'numero_normalizado',
                   'parse_status']].head(10))
print(resultado.parse_status.value_counts(dropna=False).to_string())

# Reutiliza exclusivamente a extração congelada e validada do Notebook 04.
# Este notebook não redefine nem altera o parser.
import json
from pathlib import Path

notebook04_candidates = (
    Path('notebooks/04_primary_case_number_and_collisions.ipynb'),
    Path('../notebooks/04_primary_case_number_and_collisions.ipynb'),
)
notebook04_path = next(path for path in notebook04_candidates if path.exists())
notebook04 = json.loads(notebook04_path.read_text(encoding='utf-8'))
parser_context = {'__name__': '__main__'}
for cell_index in (2, 12, 13):
    exec(''.join(notebook04['cells'][cell_index]['source']), parser_context)
resultado = parser_context['resultado'].copy()
parser_context['connection'].close()
print('Extração congelada do Notebook 04 carregada.')

# 2. Universo completo validado

Entram na análise os 1000 registros `extracted`: 200 de cada tribunal.
Os 25 casos TSE foram recuperados e auditados independentemente antes
desta análise de colisões.

In [ ]:
import unicodedata

resultado['numero_posicao'] = pd.to_numeric(resultado.numero_posicao, errors='coerce')
metricas = (resultado.pivot_table(index='tribunal', columns='parse_status',
                                  values='id', aggfunc='count', fill_value=0)
            .reindex(TRIBUNAIS, fill_value=0))
for status in ('extracted', 'ambiguous', 'suspicious', 'unparsed'):
    if status not in metricas:
        metricas[status] = 0
metricas = metricas[['extracted', 'ambiguous', 'suspicious', 'unparsed']]
metricas['total'] = metricas.sum(axis=1)
metricas['cobertura_%'] = (100 * metricas.extracted / metricas.total).round(1)
display(metricas)
assert int(metricas.extracted.sum()) == 1000
assert metricas.loc['STF', 'extracted'] == 200
assert metricas.loc['STJ', 'extracted'] == 200
assert metricas.loc['TSE', 'extracted'] == 200
assert metricas.loc['TST', 'extracted'] == 200
assert metricas.loc['STM', 'extracted'] == 200

elegiveis = resultado[resultado.parse_status.eq('extracted')].copy()
elegiveis['case_key'] = list(zip(elegiveis.tribunal, elegiveis.numero_normalizado))
elegiveis['case_key_texto'] = elegiveis.case_key.map(lambda x: f'{x[0]}|{x[1]}')
assert len(elegiveis) == 1000

# 3. Chave experimental e distribuição inicial

A chave é usada apenas como mecanismo de agrupamento. Nenhum registro é
escolhido como canônico dentro de um grupo.

In [ ]:
tamanhos = elegiveis.groupby('case_key', sort=True).id.nunique()
grupos = tamanhos.rename('quantidade_ids').reset_index()
grupos['tribunal'] = grupos.case_key.map(lambda x: x[0])
grupos['numero_normalizado'] = grupos.case_key.map(lambda x: x[1])
grupos['case_key_texto'] = grupos.case_key.map(lambda x: f'{x[0]}|{x[1]}')
multi_keys = set(grupos.loc[grupos.quantidade_ids.gt(1), 'case_key'])
multi = grupos[grupos.quantidade_ids.gt(1)].copy()
single = grupos[grupos.quantidade_ids.eq(1)].copy()
registros_multi = int(multi.quantidade_ids.sum())
resumo_chave = pd.DataFrame({
    'medida': ['registros elegíveis', 'grupos', 'grupos single-ID',
               'grupos multi-ID', 'registros em grupos multi-ID', 'maior grupo'],
    'valor': [len(elegiveis), len(grupos), len(single), len(multi),
              registros_multi, int(tamanhos.max())],
})
display(resumo_chave)
display(grupos.groupby('tribunal').agg(
    registros=('quantidade_ids', 'sum'),
    grupos=('case_key', 'size'),
    single_id=('quantidade_ids', lambda s: int(s.eq(1).sum())),
    multi_id=('quantidade_ids', lambda s: int(s.gt(1).sum())),
    registros_multi=('quantidade_ids', lambda s: int(s[s.gt(1)].sum())),
).reindex(TRIBUNAIS).reset_index())
print({'registros elegíveis': len(elegiveis), 'grupos': len(grupos),
       'single-ID': len(single), 'multi-ID': len(multi),
       'registros multi-ID': registros_multi, 'maior grupo': int(tamanhos.max())})

# 3.1. Baseline histórica e delta dos 25 TSE

A baseline abaixo é preservada como referência histórica, sem ser
recalculada por uma lógica nova: 975 registros, 889 grupos, 810
single-ID, 79 multi-ID, 165 registros em colisões e maior grupo 4.
O delta é calculado somente para os 25 documentos TSE recuperados.

In [ ]:
BASELINE_975 = {
    'registros': 975, 'grupos': 889, 'single_id': 810,
    'multi_id': 79, 'registros_multi_id': 165, 'maior_grupo': 4,
}
assert BASELINE_975 == {
    'registros': 975, 'grupos': 889, 'single_id': 810,
    'multi_id': 79, 'registros_multi_id': 165, 'maior_grupo': 4,
}

anotacoes_delta_candidates = (
    Path('data/audits/tse_25_anotacoes.json'),
    Path('../data/audits/tse_25_anotacoes.json'),
)
anotacoes_delta_path = next(path for path in anotacoes_delta_candidates if path.exists())
novos_tse_ids = {item['documento_id'] for item in json.loads(
    anotacoes_delta_path.read_text(encoding='utf-8')
)}
novos_tse = elegiveis[elegiveis.documento_id.isin(novos_tse_ids)].copy()
historico_975 = elegiveis[~elegiveis.documento_id.isin(novos_tse_ids)].copy()
assert len(novos_tse) == 25
assert len(historico_975) == 975

tamanhos_antes = historico_975.groupby(
    ['tribunal', 'numero_normalizado'], sort=True).id.nunique()
tamanhos_depois = elegiveis.groupby(
    ['tribunal', 'numero_normalizado'], sort=True).id.nunique()

def categoria_delta(row):
    chave = (row.tribunal, row.numero_normalizado)
    antes = int(tamanhos_antes.get(chave, 0))
    depois = int(tamanhos_depois.get(chave, 0))
    if antes == 0 and depois == 1:
        return 'A — nova chave single-ID'
    if antes == 1 and depois > 1:
        return 'B — single-ID virou multi-ID'
    if antes > 1 and depois > antes:
        return 'C — grupo multi-ID existente ganhou candidato'
    if antes == 0 and depois > 1:
        return 'D — colisão criada entre novos TSE'
    raise AssertionError(f'categoria inesperada para {chave}: {antes}->{depois}')

delta_registros = novos_tse.assign(
    tamanho_antes=novos_tse.apply(
        lambda r: int(tamanhos_antes.get((r.tribunal, r.numero_normalizado), 0)), axis=1),
    tamanho_depois=novos_tse.apply(
        lambda r: int(tamanhos_depois[(r.tribunal, r.numero_normalizado)]), axis=1),
)
delta_registros['categoria_delta'] = delta_registros.apply(categoria_delta, axis=1)
delta_resumo = (delta_registros.groupby('categoria_delta', as_index=False)
                .agg(registros=('id', 'size'), chaves=('case_key', 'nunique')))
delta_resumo = (delta_resumo.set_index('categoria_delta')
                .reindex(['A — nova chave single-ID',
                         'B — single-ID virou multi-ID',
                         'C — grupo multi-ID existente ganhou candidato',
                         'D — colisão criada entre novos TSE'], fill_value=0)
                .reset_index())
display(delta_resumo)

chaves_afetadas = sorted(set(zip(
    delta_registros.tribunal, delta_registros.numero_normalizado)))
def ids_para(df, chave):
    return ', '.join(df.loc[
        (df.tribunal == chave[0]) & (df.numero_normalizado == chave[1]), 'id'
    ].astype(str).tolist()) or '—'

delta_chaves = pd.DataFrame([
    {
        'tribunal': tribunal, 'numero_normalizado': numero,
        'tamanho_antes': int(tamanhos_antes.get((tribunal, numero), 0)),
        'tamanho_depois': int(tamanhos_depois[(tribunal, numero)]),
        'ids_antes': ids_para(historico_975, (tribunal, numero)),
        'ids_depois': ids_para(elegiveis, (tribunal, numero)),
        'tipo_delta': categoria_delta(delta_registros[
            (delta_registros.tribunal == tribunal) &
            (delta_registros.numero_normalizado == numero)
        ].iloc[0]),
    }
    for tribunal, numero in chaves_afetadas
])
display(delta_chaves)

# 4. Plots do universo completo

Os gráficos descrevem seletividade e concentração das colisões. Não são
julgamentos de equivalência jurídica.

In [ ]:
# Plot 2 — distribuição do tamanho dos grupos
tamanho_plot = tamanhos.map(lambda n: '1 registro' if n == 1 else (
    '2 registros' if n == 2 else ('3 registros' if n == 3 else '4+ registros'))).value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
tamanho_plot.reindex(['1 registro', '2 registros', '3 registros', '4+ registros'],
                     fill_value=0).plot.bar(ax=ax, rot=0)
ax.set_title('Distribuição do tamanho dos grupos por case_key')
ax.set_xlabel('tamanho do grupo')
ax.set_ylabel('grupos')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'A maioria dos grupos possui um registro; a cauda mostra a extensão das colisões.'
]}))

# Plot 1 — grupos multi-ID por tribunal
multi_por_tribunal = multi.groupby('tribunal').size().reindex(TRIBUNAIS, fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
multi_por_tribunal.plot.bar(ax=ax, rot=0)
ax.set_title('Grupos multi-ID por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('grupos multi-ID')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'A concentração por tribunal indica onde a chave exige maior desambiguação.'
]}))

# Plot 3 — baseline 975 versus universo completo
baseline_plot = pd.DataFrame({
    'single-ID': [BASELINE_975['single_id'], len(single)],
    'multi-ID': [BASELINE_975['multi_id'], len(multi)],
    'registros em colisões': [BASELINE_975['registros_multi_id'], registros_multi],
}, index=['975 registros', '1000 registros'])
fig, ax = plt.subplots(figsize=(7, 4))
baseline_plot.plot.bar(ax=ax, rot=0)
ax.set_title('Baseline 975 versus universo completo')
ax.set_xlabel('universo')
ax.set_ylabel('quantidade')
fig.tight_layout()
plt.show()
display(baseline_plot)
display(pd.DataFrame({'interpretação': [
    'Os 25 TSE aumentam grupos e candidatos em colisões; a chave continua sendo de recuperação, não de identidade única.'
]}))

# 5. Evidência de todos os grupos multi-ID

A tabela abaixo contém todos os candidatos dos grupos multi-ID do universo
trechos são compactos para inspeção; o texto completo não é impresso.
`texto_exatamente_idêntico` é um fato documental, não uma decisão sobre
identidade jurídica.

In [ ]:
elegiveis['texto_hash'] = elegiveis.texto.map(lambda x: hashlib.sha256(x.encode()).hexdigest())

def normalizar_comparacao(value):
    if pd.isna(value) or not str(value).strip():
        return None
    sem_acentos = unicodedata.normalize('NFKD', str(value)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^A-Z0-9]+', ' ', sem_acentos.upper()).strip()

def status_campo(grupo, campo):
    valores = grupo[campo].dropna().map(str)
    if len(valores) == len(grupo) and valores.nunique() == 1:
        return 'iguais'
    if valores.nunique() > 1:
        return 'diferentes'
    return 'indisponível'

evidence_state_names = {
    'ACRE': 'AC', 'ALAGOAS': 'AL', 'AMAPÁ': 'AP', 'AMAPA': 'AP',
    'AMAZONAS': 'AM', 'BAHIA': 'BA', 'CEARÁ': 'CE', 'CEARA': 'CE',
    'DISTRITO FEDERAL': 'DF', 'ESPÍRITO SANTO': 'ES', 'ESPIRITO SANTO': 'ES',
    'GOIÁS': 'GO', 'GOIAS': 'GO', 'MARANHÃO': 'MA', 'MARANHAO': 'MA',
    'MATO GROSSO': 'MT', 'MATO GROSSO DO SUL': 'MS', 'MINAS GERAIS': 'MG',
    'PARÁ': 'PA', 'PARA': 'PA', 'PARAÍBA': 'PB', 'PARAIBA': 'PB',
    'PARANÁ': 'PR', 'PARANA': 'PR', 'PERNAMBUCO': 'PE', 'PIAUÍ': 'PI',
    'PIAUI': 'PI', 'RIO DE JANEIRO': 'RJ', 'RIO GRANDE DO NORTE': 'RN',
    'RIO GRANDE DO SUL': 'RS', 'RONDÔNIA': 'RO', 'RONDONIA': 'RO',
    'RORAIMA': 'RR', 'SANTA CATARINA': 'SC', 'SÃO PAULO': 'SP',
    'SAO PAULO': 'SP', 'SERGIPE': 'SE', 'TOCANTINS': 'TO',
}

def trecho_numero(bloco, row):
    if not isinstance(bloco, str):
        return None
    if row.tribunal == 'TSE':
        return CNJ.search(bloco)
    if row.tribunal == 'STJ':
        return STJ_TITLE.search(bloco)
    if row.tribunal == 'STM':
        return STM_TITLE.search(bloco)
    if row.numero_raw:
        pos = bloco.find(row.numero_raw)
        if pos >= 0:
            return re.search(re.escape(row.numero_raw), bloco)
    return None

def metadados_formais(row):
    bloco = row.identity_block if isinstance(row.identity_block, str) else ''
    classe_raw = None
    uf = None
    numero_match = trecho_numero(bloco, row)
    numero_inicio = numero_match.start() if numero_match else -1
    if row.tribunal == 'STJ':
        match = STJ_TITLE.search(bloco)
        if match:
            classe_raw = match.group('class').strip()
            uf = match.group('uf').upper()
    elif row.tribunal == 'STM':
        match = STM_TITLE.search(bloco)
        if match:
            classe_raw = match.group('class').strip()
    elif row.tribunal == 'STF' and numero_inicio >= 0:
        prefixo = bloco[:numero_inicio]
        prefixo = re.sub(
            r'^(?:PRIMEIRA|SEGUNDA|TERCEIRA|QUARTA|QUINTA|SEXTA) TURMA|^PLEN[ÁA]RIO',
            '', prefixo, flags=re.I).strip()
        classe_raw = prefixo or None
    elif row.tribunal == 'TST':
        match = re.search(r'ESTES AUTOS DE\s+(.*?)\s+N[ºO.]', bloco, re.I)
        if match:
            classe_raw = match.group(1).strip()
    elif row.tribunal == 'TSE' and numero_inicio >= 0:
        cabecalho = re.match(
            r'TR(?:IB|IE)UNAL SUPERIOR ELEITORAL.*?AC[ÓO€]RD[ÃA]O',
            bloco[:numero_inicio], flags=re.I)
        if cabecalho:
            classe_raw = bloco[cabecalho.end():numero_inicio].strip()

    if numero_match:
        cauda = bloco[numero_match.end():numero_match.end() + 100]
        uf_match = re.match(r'\s*/\s*([A-Z]{2})', cauda, re.I)
        if uf_match:
            uf = uf_match.group(1).upper()
        elif uf is None:
            cauda_normalizada = normalizar_comparacao(cauda) or ''
            for estado, sigla in sorted(evidence_state_names.items(), key=lambda x: -len(x[0])):
                if normalizar_comparacao(estado) in cauda_normalizada:
                    uf = sigla
                    break
    return pd.Series({
        'classe_raw': classe_raw,
        'classe_normalizada': normalizar_comparacao(classe_raw),
        'uf': uf,
    })

elegiveis = pd.concat([elegiveis, elegiveis.apply(metadados_formais, axis=1)], axis=1)
registros_multi = elegiveis[elegiveis.case_key.isin(multi_keys)].copy()
resumo_grupo = (registros_multi.groupby('case_key', sort=True)
                .agg(tribunal=('tribunal', 'first'),
                     numero_raw=('numero_raw', 'first'),
                     numero_normalizado=('numero_normalizado', 'first'),
                     quantidade_ids=('id', 'nunique'),
                     textos=('texto', 'nunique'),
                     hashes=('texto_hash', 'nunique'))
                .reset_index())
resumo_grupo['texto_exatamente_idêntico'] = resumo_grupo.textos.eq(1)
registros_multi = registros_multi.merge(
    resumo_grupo[['case_key', 'quantidade_ids', 'texto_exatamente_idêntico']],
    on='case_key', how='left')
registros_multi['trecho_bloco_formal'] = registros_multi.identity_block.map(lambda x: preview(x, 520))
registros_multi['tamanho_texto'] = registros_multi.texto.str.len()
evidencia_colisoes = registros_multi[[
    'case_key', 'tribunal', 'numero_raw', 'numero_normalizado', 'quantidade_ids',
    'id', 'documento_id', 'ano', 'relator', 'trecho_bloco_formal',
    'classe_raw', 'classe_normalizada', 'uf', 'tamanho_texto', 'texto_hash',
    'texto_exatamente_idêntico',
]].sort_values(['tribunal', 'numero_normalizado', 'id'])
assert len(evidencia_colisoes) == registros_multi.id.nunique()
display(evidencia_colisoes)

# 6. Duplicatas exatas

Textos byte-a-byte idênticos são medidos primeiro. Isso é evidência forte
de duplicação documental, mas permanece uma hipótese documental e não é
convertido em decisão final de feito.

In [ ]:
duplicatas_exatas = resumo_grupo[resumo_grupo.texto_exatamente_idêntico].copy()
duplicata_resumo = pd.DataFrame({
    'medida': ['grupos com texto idêntico', 'IDs envolvidos', 'grupos com texto diferente'],
    'quantidade': [len(duplicatas_exatas),
                   int(duplicatas_exatas.quantidade_ids.sum()),
                   int((~resumo_grupo.texto_exatamente_idêntico).sum())],
})
display(duplicata_resumo)
display(duplicatas_exatas.groupby('tribunal').agg(
    grupos=('case_key', 'size'), ids=('quantidade_ids', 'sum')
).reindex(TRIBUNAIS, fill_value=0).reset_index())
display(evidencia_colisoes[evidencia_colisoes.texto_exatamente_idêntico]
        [['case_key', 'tribunal', 'id', 'documento_id', 'texto_hash']]
        .groupby('case_key', as_index=False).head(2).head(12))

plot_duplicatas = pd.Series({
    'texto exatamente idêntico': int(duplicatas_exatas.shape[0]),
    'texto diferente': int((~resumo_grupo.texto_exatamente_idêntico).sum()),
})
fig, ax = plt.subplots(figsize=(7, 4))
plot_duplicatas.plot.bar(ax=ax, rot=0)
ax.set_title('Grupos multi-ID com e sem texto idêntico')
ax.set_xlabel('evidência textual')
ax.set_ylabel('grupos multi-ID')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'Texto idêntico caracteriza duplicação documental provável, sem resolver a identidade jurídica.'
]}))

# 7. Comparação objetiva das evidências

A tabela resume igualdade, diferença ou indisponibilidade dentro de cada
grupo multi-ID. Valores ausentes não são tratados como iguais.

In [ ]:
comparacoes = []
for campo in ('texto_hash', 'classe_normalizada', 'uf', 'relator', 'ano'):
    if campo == 'texto_hash':
        iguais = int(resumo_grupo.texto_exatamente_idêntico.sum())
        diferentes = int((~resumo_grupo.texto_exatamente_idêntico).sum())
        indisponivel = 0
    else:
        estados = registros_multi.groupby('case_key', sort=True).apply(
            lambda grupo: status_campo(grupo, campo), include_groups=False)
        iguais = int(estados.eq('iguais').sum())
        diferentes = int(estados.eq('diferentes').sum())
        indisponivel = int(estados.eq('indisponível').sum())
    comparacoes.append({'evidência': campo, 'grupos iguais': iguais,
                        'grupos diferentes': diferentes,
                        'indisponível': indisponivel})
tabela_evidencias = pd.DataFrame(comparacoes)
display(tabela_evidencias)

# 8. Classe dentro das colisões

A classe é extraída somente do bloco formal dos candidatos em colisão. A
normalização remove acentos e pontuação sem tentar impor uma taxonomia
jurídica. A diferença de classe é apenas evidência descritiva.

In [ ]:
classe_por_grupo = (registros_multi.groupby('case_key', sort=True)
                    .apply(lambda grupo: status_campo(grupo, 'classe_normalizada'),
                           include_groups=False)
                    .rename('status_classe').reset_index())
display(pd.DataFrame({
    'medida': ['grupos com classe disponível', 'mesma classe em todos',
               'classes diferentes', 'sem classe completa'],
    'quantidade': [int(classe_por_grupo.status_classe.ne('indisponível').sum()),
                   int(classe_por_grupo.status_classe.eq('iguais').sum()),
                   int(classe_por_grupo.status_classe.eq('diferentes').sum()),
                   int(classe_por_grupo.status_classe.eq('indisponível').sum())],
}))

# 9. UF, relator e ano como evidência auxiliar

UF não é adicionada à chave. Relator e ano também não são usados como
chaves: a análise apenas mede coincidências e divergências dentro das
colisões.

In [ ]:
aux_resumo = tabela_evidencias[tabela_evidencias.evidência.isin(
    ['uf', 'relator', 'ano'])].copy()
display(aux_resumo)

# 10. Inspeção compacta de todos os grupos multi-ID

Cada linha representa um candidato e preserva o conjunto completo de
grupos multi-ID para inspeção posterior, sem imprimir documentos inteiros.

In [ ]:
inspecao_todos_grupos = evidencia_colisoes[[
    'case_key', 'tribunal', 'numero_raw', 'id', 'documento_id',
    'classe_normalizada', 'uf', 'relator', 'ano',
    'texto_exatamente_idêntico', 'trecho_bloco_formal',
]].copy()
display(inspecao_todos_grupos)

# 10.1. Auditoria do delta TSE

Somente as chaves novas multi-ID são auditadas nesta etapa. Os dois
grupos abaixo foram inspecionados manualmente pela igualdade textual,
bloco formal, classe, UF, relator e ano. O julgamento é uma anotação
explícita da auditoria, não uma inferência automática do parser.

In [ ]:
delta_multi_keys = set(zip(
    delta_chaves.loc[delta_chaves.tamanho_depois.gt(1), 'tribunal'],
    delta_chaves.loc[delta_chaves.tamanho_depois.gt(1), 'numero_normalizado']))

def metadados_delta(row):
    bloco = row.trecho_bloco_formal or ''
    inicio_numero = bloco.find(row.numero_raw or '')
    prefixo = bloco[:inicio_numero] if inicio_numero >= 0 else bloco
    marcador = re.search(r'AC(?:Ó|O|€)\s*RD(?:Ã|A)?\s*O', prefixo, re.I)
    classe = prefixo[marcador.end():].strip() if marcador else None
    cauda = bloco[inicio_numero + len(row.numero_raw or ''):] if inicio_numero >= 0 else bloco
    uf = None
    cauda_normalizada = normalizar_comparacao(cauda) or ''
    for estado, sigla in sorted(evidence_state_names.items(), key=lambda item: -len(item[0])):
        if normalizar_comparacao(estado) in cauda_normalizada:
            uf = sigla
            break
    return pd.Series({'classe_delta': classe, 'uf_delta': uf})

evidencia_delta = evidencia_colisoes[
    evidencia_colisoes.case_key.isin(delta_multi_keys)
].copy()
evidencia_delta = pd.concat([
    evidencia_delta, evidencia_delta.apply(metadados_delta, axis=1)
], axis=1)

# Julgamentos manuais explícitos para as duas chaves novas; não são regras do parser.
julgamentos_delta = {
    ('TSE', '1028120126200019'): (
        'same_case', 'textos byte-a-byte idênticos; mesmo bloco formal e metadados'),
    ('TSE', '4774520126260139'): (
        'same_case', 'textos byte-a-byte idênticos; mesmo bloco formal e metadados'),
}
assert delta_multi_keys == set(julgamentos_delta)

auditoria_delta = (evidencia_delta.groupby('case_key', sort=True)
    .apply(lambda grupo: pd.Series({
        'tribunal': grupo.tribunal.iloc[0],
        'numero': grupo.numero_normalizado.iloc[0],
        'candidatos': '; '.join(
            f'{r.id}/{r.documento_id}' for r in grupo.itertuples()),
        'bloco_formal': ' || '.join(grupo.trecho_bloco_formal),
        'classe': ' || '.join(grupo.classe_delta.fillna('indisponível')),
        'uf': ' || '.join(grupo.uf_delta.fillna('indisponível')),
        'relator': ' || '.join(grupo.relator.fillna('indisponível')),
        'ano': ' || '.join(grupo.ano.astype(str)),
        'texto_exatamente_idêntico': bool(grupo.texto_exatamente_idêntico.all()),
        'label': julgamentos_delta[grupo.name][0],
        'justificativa': julgamentos_delta[grupo.name][1],
    }), include_groups=False)).reset_index()
display(auditoria_delta)
assert set(auditoria_delta.label) == {'same_case'}

colisoes_finais = pd.DataFrame({
    'label': ['same_case', 'different_case', 'indeterminate'],
    'grupos': [79 + len(auditoria_delta), 0, 0],
    'candidatos': [165 + int(evidencia_delta.id.nunique()), 0, 0],
})
display(colisoes_finais)

# 11. Taxonomia descritiva das colisões

As categorias abaixo descrevem somente as evidências observadas. Nenhuma
coluna de `same_case`, `different_case` ou `resolved` é criada.

In [ ]:
estado_por_grupo = {}
for _, grupo in registros_multi.groupby('case_key', sort=True):
    estados = {campo: status_campo(grupo, campo)
               for campo in ('classe_normalizada', 'uf', 'relator', 'ano')}
    texto_igual = grupo.texto.nunique(dropna=False) == 1
    if texto_igual:
        categoria = 'A — texto exatamente idêntico'
    elif estados['classe_normalizada'] == 'diferentes':
        categoria = 'C — classe diferente'
    elif estados['uf'] == 'diferentes':
        categoria = 'D — UF diferente'
    elif estados['relator'] == 'diferentes' or estados['ano'] == 'diferentes':
        categoria = 'E — relator/ano diferente'
    elif all(value == 'iguais' for value in estados.values()):
        categoria = 'B — texto diferente, metadados estruturais iguais'
    elif all(value == 'indisponível' for value in estados.values()):
        categoria = 'G — informação insuficiente'
    else:
        categoria = 'F — evidências mistas'
    estado_por_grupo[grupo.case_key.iloc[0]] = categoria
resumo_grupo['categoria_descritiva'] = resumo_grupo.case_key.map(estado_por_grupo)
categorias_colisoes = (resumo_grupo.groupby('categoria_descritiva', as_index=False)
                       .agg(grupos=('case_key', 'size'),
                            registros=('quantidade_ids', 'sum')))
display(categorias_colisoes)
fig, ax = plt.subplots(figsize=(9, 4))
categorias_colisoes.set_index('categoria_descritiva').grupos.plot.bar(ax=ax, rot=20)
ax.set_title('Distribuição das categorias descritivas de colisão')
ax.set_xlabel('categoria de evidência')
ax.set_ylabel('grupos')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'As categorias organizam evidências para auditoria; não afirmam same_case ou different_case.'
]}))

# 12. Seletividade da chave como mecanismo de recuperação

A chave deve recuperar poucos candidatos. Esse diagnóstico não a promove
a chave final de identidade.

In [ ]:
seletividade = pd.DataFrame({
    'medida': ['grupos single-ID', 'grupos multi-ID', 'percentual single-ID',
               'percentual multi-ID', 'tamanho médio dos grupos multi-ID', 'maior grupo'],
    'valor': [len(single), len(multi), round(100 * len(single) / len(grupos), 1),
              round(100 * len(multi) / len(grupos), 1),
              round(multi.quantidade_ids.mean(), 2), int(tamanhos.max())],
})
display(seletividade)
candidatos_plot = tamanhos.map(lambda n: '1 candidato' if n == 1 else (
    '2 candidatos' if n == 2 else ('3 candidatos' if n == 3 else '4+ candidatos'))).value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
candidatos_plot.reindex(['1 candidato', '2 candidatos', '3 candidatos', '4+ candidatos'],
                        fill_value=0).plot.bar(ax=ax, rot=0)
ax.set_title('Número de candidatos por case_key')
ax.set_xlabel('candidatos recuperados')
ax.set_ylabel('grupos')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'A chave é seletiva para recuperação quando retorna um grupo pequeno, mas grupos multi-ID exigem evidência adicional.'
]}))

# 13. Comparação incremental nas colisões

Classe e UF são testadas apenas quando estão disponíveis para todos os
candidatos do grupo-base. Reduzir colisões não equivale a provar melhor
identidade; a perda de cobertura é mostrada junto.

In [ ]:
def estrategia(longo, colunas):
    if not colunas:
        validos = pd.Series(True, index=grupos.case_key)
        registros_com_metadado = len(elegiveis)
    else:
        validos = (elegiveis.groupby('case_key')[colunas]
                   .apply(lambda grupo: grupo.notna().all().all(), include_groups=False))
        registros_com_metadado = int(elegiveis[colunas].notna().all(axis=1).sum())
    chaves_validas = set(validos[validos].index)
    avaliaveis = elegiveis[elegiveis.case_key.isin(chaves_validas)].copy()
    if colunas:
        avaliaveis['chave_refinada'] = avaliaveis.case_key_texto + '|' + avaliaveis[colunas].astype(str).agg('|'.join, axis=1)
    else:
        avaliaveis['chave_refinada'] = avaliaveis.case_key_texto
    refinados = avaliaveis.groupby('chave_refinada').id.nunique()
    multi_base_avaliaveis = int(multi.case_key.isin(chaves_validas).sum())
    multi_restantes = int(refinados.gt(1).sum())
    return {
        'estratégia': longo,
        'cobertura_registros': registros_com_metadado,
        'cobertura_%': round(100 * registros_com_metadado / len(elegiveis), 1),
        'grupos_avaliáveis': len(chaves_validas),
        'grupos_multi-ID_restantes': multi_restantes,
        'grupos_separados': multi_base_avaliaveis - multi_restantes,
        'grupos_multi-ID_sem_metadado': len(multi) - multi_base_avaliaveis,
    }

estrategias = pd.DataFrame([
    estrategia('A — tribunal + número', []),
    estrategia('B — tribunal + número + classe', ['classe_normalizada']),
    estrategia('C — tribunal + número + UF', ['uf']),
    estrategia('D — tribunal + número + classe + UF', ['classe_normalizada', 'uf']),
])
display(estrategias)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
estrategias.plot.bar(x='estratégia', y='grupos_multi-ID_restantes', ax=axes[0], legend=False, rot=25)
axes[0].set_title('Colisões restantes por estratégia')
axes[0].set_xlabel('estratégia')
axes[0].set_ylabel('grupos multi-ID')
estrategias.plot.bar(x='estratégia', y='cobertura_%', ax=axes[1], legend=False, rot=25)
axes[1].set_title('Cobertura por estratégia')
axes[1].set_xlabel('estratégia')
axes[1].set_ylabel('% dos registros elegíveis')
fig.tight_layout()
plt.show()
display(pd.DataFrame({'interpretação': [
    'Estratégias incrementais expõem trade-off entre metadado disponível e colisões remanescentes; não definem a chave final.'
]}))

# 14. Múltiplos IDs para uma identidade recuperável

Quando uma chave aponta para textos idênticos ou para registros com a mesma
identidade formal, a análise preserva todos os `id` candidatos. A decisão
futura deve considerar uma lista de candidatos, nunca selecionar um ID
arbitrariamente.

In [ ]:
multiplos_ids_evidencia = (resumo_grupo.assign(
    mesma_identidade_formal= resumo_grupo.case_key.map(
        classe_por_grupo.set_index('case_key').status_classe.eq('iguais')
    )
))
display(multiplos_ids_evidencia[
    ['case_key', 'quantidade_ids', 'texto_exatamente_idêntico', 'mesma_identidade_formal',
     'categoria_descritiva']
])
print('Todos os candidatos permanecem representados por id/documento_id; nenhum id é escolhido.')

# 15. Hipótese operacional, sem implementação

## Chave de recuperação

A hipótese de trabalho continua sendo `(tribunal, numero_normalizado)`.

## Evidências de desambiguação

Classe, UF, bloco formal, relator, ano e igualdade textual podem orientar
uma etapa posterior de auditoria. Nenhum deles é adicionado
automaticamente à chave nesta etapa.

## Representação dos candidatos

Conceitualmente, a recuperação deve devolver todos os candidatos associados
à chave, preservando `id`, `documento_id` e metadados disponíveis. Essa
estrutura é apenas hipótese; `CanonicalIndex` não é implementado aqui.

## Resultado desta etapa

As 81 colisões multi-ID estão semanticamente compreendidas: 79 grupos
antigos reutilizam a auditoria `same_case` e as 2 colisões novas do delta
foram auditadas como `same_case`. O notebook preserva múltiplos candidatos
e não implementa `CanonicalIndex`, resolver ou detector.

# 16. Auditoria manual incorporada — 25 casos TSE

As anotações manuais ficam em `data/audits/tse_25_anotacoes.json`. Esta
seção apenas lê o arquivo, valida sua cobertura e exibe os trechos
delimitados por `<inicio>` e `<fim>`. Ela não cria rótulos, não altera o
parser e não substitui a decisão da auditoria independente.

In [ ]:
import json
from pathlib import Path

anotacoes_tse_candidates = (
    Path('data/audits/tse_25_anotacoes.json'),
    Path('../data/audits/tse_25_anotacoes.json'),
)
anotacoes_tse_path = next(path for path in anotacoes_tse_candidates if path.exists())
anotacoes_tse = json.loads(anotacoes_tse_path.read_text(encoding='utf-8'))
assert len(anotacoes_tse) == 25
assert {item['tribunal'] for item in anotacoes_tse} == {'TSE'}
assert {item['parse_status'] for item in anotacoes_tse} == {'suspicious', 'unparsed'}

def extrair_trechos_manuais(anotacao):
    return [trecho.strip() for trecho in re.findall(
        r'<inicio>(.*?)<fim>', anotacao or '', flags=re.DOTALL
    )]

anotacoes_tse_df = pd.DataFrame([
    {
        'documento_id': item['documento_id'],
        'id': item['id'],
        'tribunal': item['tribunal'],
        'parse_status': item['parse_status'],
        'quantidade_trechos': len(extrair_trechos_manuais(item.get('anotacao'))),
        'trechos_manuais': '\n---\n'.join(extrair_trechos_manuais(item.get('anotacao'))),
    }
    for item in anotacoes_tse
])

assert anotacoes_tse_df.documento_id.is_unique
assert set(anotacoes_tse_df.documento_id).issubset(
    set(resultado.loc[resultado.tribunal.eq('TSE'), 'documento_id'])
)
display(anotacoes_tse_df[['documento_id', 'id', 'parse_status',
                         'quantidade_trechos', 'trechos_manuais']])
print('Anotações manuais carregadas:', len(anotacoes_tse_df))
print('Trechos delimitados:', int(anotacoes_tse_df.quantidade_trechos.sum()))

# 17. Findings

## Universo analisado

Foram analisados 1000 registros `extracted`: STF 200, STJ 200, TSE 200,
TST 200 e STM 200. Os 25 casos TSE foram recuperados e auditados.

## Colisões

A chave experimental forma 912 grupos, dos quais 831 são single-ID e 81
são multi-ID, envolvendo 169 registros; o maior grupo tem 4 candidatos.
O delta dos 25 TSE criou 21 chaves single-ID e 2 colisões novas entre
registros TSE; não houve entrada em colisão antiga.

## Duplicatas exatas

Texto idêntico é evidência documental forte e deve ser preservado como
hipótese de duplicação, sem virar regra automática de identidade jurídica.

## Classe, UF e outros metadados

Classe, UF, relator e ano são evidências auxiliares medidas somente dentro
das colisões. Diferenças reduzem a plausibilidade de equivalência, mas não
recebem interpretação automática.

## `(tribunal, número)`

A chave é seletiva para recuperar conjuntos pequenos, mas não resolve todos
os grupos multi-ID. A redução incremental por classe/UF deve ser lida junto
com sua cobertura.

## Múltiplos IDs

As 79 colisões antigas permanecem `same_case`; as 2 colisões novas do
delta TSE também foram auditadas como `same_case`. Todos os candidatos
foram preservados; não há escolha arbitrária de ID canônico.

## Questão aberta

Não restam colisões `indeterminate` relevantes no universo auditado. A
chave `(tribunal, numero_normalizado)` é adequada para recuperação, mas
não é uma identidade única. `CanonicalIndex` não é implementado aqui.

In [ ]:
print('Resumo da chave experimental:')
display(resumo_chave)
print('Delta dos 25 TSE:')
display(delta_resumo)
print('Resultado final das colisões auditadas:')
display(colisoes_finais)
print('Exploração estrutural encerrada; CanonicalIndex não implementado.')
connection.close()
print('Conexão read-only encerrada.')